# distributed-sampler-shard — worked example 2: Show that set_epoch changes the permutation across epochs

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `distributed-sampler-shard`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Calling `sampler.set_epoch(epoch)` before iterating is required for correct distributed training: it seeds the internal RNG differently each epoch so each rank sees a different permutation and no rank gets stuck with the same samples in the same order. If `set_epoch` is not called, the shuffle is the same every epoch — breaking the statistical independence between epochs.

## Worked solution

We create one sampler per epoch (equivalent to calling `set_epoch(epoch)` on the same sampler).

**Epoch 0:** `sampler.set_epoch(0)` → indices e.g. `[3, 7, 1, 5, ...]`.
**Epoch 1:** `sampler.set_epoch(1)` → different permutation, e.g. `[6, 2, 8, 0, ...]`.

The permutations differ because the epoch number is mixed into the random seed. After enough epochs, every sample appears roughly equally often at each rank.

**Key check:** `list(sampler_epoch_0) != list(sampler_epoch_1)` (with overwhelmingly high probability for any non-trivial dataset).

In [ ]:
import torch
from torch.utils.data import TensorDataset
from torch.utils.data.distributed import DistributedSampler

def show_epoch_permutation_change(n_samples=20, world_size=2, num_epochs=3, seed=0):
    dataset = TensorDataset(torch.arange(n_samples))

    epoch_shards = []
    for epoch in range(num_epochs):
        rank_shards = []
        for rank in range(world_size):
            sampler = DistributedSampler(
                dataset, num_replicas=world_size, rank=rank,
                shuffle=True, seed=seed
            )
            sampler.set_epoch(epoch)
            rank_shards.append(list(sampler))
        epoch_shards.append(rank_shards)

    print(f'Dataset size: {n_samples}, world_size: {world_size}')
    for epoch in range(num_epochs):
        print(f'Epoch {epoch}, rank-0 indices: {epoch_shards[epoch][0]}')

    # Check that consecutive epochs produce different permutations (rank 0)
    for e in range(num_epochs - 1):
        same = epoch_shards[e][0] == epoch_shards[e + 1][0]
        print(f'Epoch {e} == Epoch {e+1} for rank 0: {same}  (should be False)')

    return epoch_shards

result = show_epoch_permutation_change()